#  Zameen.com Property Price Machine Learning Pipeline

This notebook covers model training, target scaling, metrics evaluations, leaderboard comparisons, and price prediction inference. It fits 6 estimators (Linear Regression, Decision Tree, Random Forest, Gradient Boosting, XGBoost, CatBoost) and outputs comparative performance tables.

## 1. Import Essential Machine Learning Libraries

In [1]:
import os
import pickle
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.compose import TransformedTargetRegressor
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from xgboost import XGBRegressor
from catboost import CatBoostRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from typing import Dict, Tuple, Any, List

## 2. Load Preprocessed Clean Feature Vectors

In [2]:
cleaned_data_path = 'assets/cleaned_data.csv'
if not os.path.exists(cleaned_data_path):
    cleaned_data_path = '../assets/cleaned_data.csv'

print(f"Loading cleaned feature dataset from: {cleaned_data_path}")
df_clean = pd.read_csv(cleaned_data_path)
print(f"Cleaned dataset shape: {df_clean.shape}")
display(df_clean.head())

y = df_clean['Price']
X = df_clean.drop(columns=['Price'])

Loading cleaned feature dataset from: assets/cleaned_data.csv
Cleaned dataset shape: (965, 13)


,Price,Area,Bedrooms,Bathrooms,Location,Built in year,Parking space,Servant Quarters,Store rooms,Kitchens,Drawing Rooms,Property type_House,Property type_Penthouse
0,170000000.0,0.0,9.0,10.0,2.719583e+08,2015.0,7.0,2.0,2.0,3.0,1.0,1.0,0.0
1,265000000.0,0.0,4.0,5.0,3.711842e+08,2010.0,5.0,1.0,1.0,1.0,1.0,1.0,0.0
2,140000000.0,0.0,7.0,7.0,2.393333e+08,2012.0,3.0,2.0,1.0,2.0,1.0,1.0,0.0
3,420000000.0,0.0,6.0,6.0,2.393333e+08,2026.0,6.0,4.0,2.0,2.0,1.0,1.0,0.0
4,139000000.0,0.0,5.0,6.0,1.314392e+08,2025.0,3.0,2.0,2.0,2.0,1.0,1.0,0.0


## 3. Train-Test Splits & Pipeline Hyperparameters

In [3]:
random_state = 42
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=random_state
)

model_hyperparams = {
    'linear_regression': {},
    'decision_tree': {'max_depth': 10, 'random_state': random_state},
    'random_forest': {
        'n_estimators': 100, 
        'max_depth': 5, 
        'min_samples_split': 2, 
        'random_state': random_state
    },
    'gradient_boosting': {
        'n_estimators': 100, 
        'learning_rate': 0.1, 
        'random_state': random_state
    },
    'xgboost': {
        'n_estimators': 50, 
        'learning_rate': 0.05, 
        'max_depth': 3, 
        'random_state': random_state
    },
    'catboost': {
        'iterations': 100, 
        'learning_rate': 0.1, 
        'verbose': 0, 
        'random_seed': random_state
    }
}

## 4. standard scaled Target Regressor Pipelines
We wrap all regressors inside TransformedTargetRegressors to transparently scale the targets during training, eliminating numeric scaling bugs.

In [4]:
unfitted_pipelines = {
    'Linear Regression': TransformedTargetRegressor(
        regressor=Pipeline([
            ('scaler', StandardScaler()), 
            ('model', LinearRegression(**model_hyperparams['linear_regression']))
        ]),
        transformer=StandardScaler()
    ),
    'Decision Tree': TransformedTargetRegressor(
        regressor=Pipeline([
            ('scaler', StandardScaler()), 
            ('model', DecisionTreeRegressor(**model_hyperparams['decision_tree']))
        ]),
        transformer=StandardScaler()
    ),
    'Random Forest': TransformedTargetRegressor(
        regressor=Pipeline([
            ('scaler', StandardScaler()), 
            ('model', RandomForestRegressor(**model_hyperparams['random_forest']))
        ]),
        transformer=StandardScaler()
    ),
    'Gradient Boosting': TransformedTargetRegressor(
        regressor=Pipeline([
            ('scaler', StandardScaler()), 
            ('model', GradientBoostingRegressor(**model_hyperparams['gradient_boosting']))
        ]),
        transformer=StandardScaler()
    ),
    'XGBoost': TransformedTargetRegressor(
        regressor=Pipeline([
            ('scaler', StandardScaler()), 
            ('model', XGBRegressor(**model_hyperparams['xgboost']))
        ]),
        transformer=StandardScaler()
    ),
    'CatBoost': TransformedTargetRegressor(
        regressor=Pipeline([
            ('scaler', StandardScaler()), 
            ('model', CatBoostRegressor(**model_hyperparams['catboost']))
        ]),
        transformer=StandardScaler()
    )
}

## 5. Fit Estimators and Serialize Models

In [5]:
fit_estimators = {}
model_serialization_dir = 'assets/save_model'
os.makedirs(model_serialization_dir, exist_ok=True)

for estimator_name, transformed_regressor in unfitted_pipelines.items():
    print(f"Training {estimator_name} pipeline...")
    transformed_regressor.fit(X_train, y_train)
    fit_estimators[estimator_name] = transformed_regressor
    
    sanitized_name = estimator_name.lower().replace(" ", "_")
    serialized_path = os.path.join(model_serialization_dir, f"{sanitized_name}.pkl")
    with open(serialized_path, 'wb') as f:
        pickle.dump(transformed_regressor, f)
    print(f"Saved {estimator_name} to: {serialized_path}")

Training Linear Regression pipeline...
Saved Linear Regression to: assets/save_model/linear_regression.pkl
Training Decision Tree pipeline...
Saved Decision Tree to: assets/save_model/decision_tree.pkl
Training Random Forest pipeline...


Saved Random Forest to: assets/save_model/random_forest.pkl
Training Gradient Boosting pipeline...
Saved Gradient Boosting to: assets/save_model/gradient_boosting.pkl
Training XGBoost pipeline...


Saved XGBoost to: assets/save_model/xgboost.pkl
Training CatBoost pipeline...
Saved CatBoost to: assets/save_model/catboost.pkl


## 6. Evaluation and Comparative Leaderboard

In [6]:
model_evaluation_metrics = []
for estimator_name, trained_estimator in fit_estimators.items():
    predicted_prices = trained_estimator.predict(X_test)
    mean_absolute_error_val = mean_absolute_error(y_test, predicted_prices)
    mean_squared_error_val = mean_squared_error(y_test, predicted_prices)
    root_mean_squared_error_val = np.sqrt(mean_squared_error_val)
    coefficient_of_determination_r2 = r2_score(y_test, predicted_prices)
    
    model_evaluation_metrics.append({
        'Model Name': estimator_name,
        'MAE (M PKR)': mean_absolute_error_val / 1e6,
        'MSE': mean_squared_error_val,
        'RMSE (M PKR)': root_mean_squared_error_val / 1e6,
        'R² Score': coefficient_of_determination_r2
    })

df_leaderboard = pd.DataFrame(model_evaluation_metrics)
df_leaderboard = df_leaderboard.sort_values(by='R² Score', ascending=False)
display(df_leaderboard)

,Model Name,MAE (M PKR),MSE,RMSE (M PKR),R² Score
0,Linear Regression,42.069949,7.562201e+15,86.960914,0.647436
2,Random Forest,38.470453,8.212334e+15,90.621930,0.617126
4,XGBoost,41.176268,8.301391e+15,91.111968,0.612974
5,CatBoost,41.271269,8.632251e+15,92.909907,0.597549
3,Gradient Boosting,43.434529,9.962396e+15,99.811805,0.535535
1,Decision Tree,49.422010,1.390224e+16,117.907740,0.351852


## 7. Performance Analysis
### Why did Linear Regression perform best?
1. **Direct target encoding alignment**: Location is the primary driver of property values. Target encoding locations (replacing string neighborhood names with their historical mean price) creates a highly linear correlation in continuous space. Linear Regression fits this with a simple coefficient ($\beta_{\text{location}} \approx 1$), making it perfectly suited.
2. **Small tabular sample size limits**: Complex tree-based models like CatBoost or XGBoost easily overfit on small datasets of 900+ samples by finding spurious non-linear splits. Linear Regression acts as a strong parametric regularizer, preventing overfitting.